# Regional Trends & Price Elasticity
**Datathon 2026 - Geographic & Economics Deep-Dive**

This notebook investigates if the 2018-2019 collapse was uniform across regions and if pricing changes impacted conversion rates.

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Project configuration
PROCESSED_DIR = Path("../data/processed")
PLOT_DIR = Path("../data/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
pd.options.display.max_columns = None

## 0. Data Loading & Date Processing
Loading all necessary tables for the deep-dive.

In [14]:
print("Loading datasets...")
orders = pd.read_parquet(PROCESSED_DIR / "orders.parquet")
geo = pd.read_parquet(PROCESSED_DIR / "geography.parquet")
order_items = pd.read_parquet(PROCESSED_DIR / "order_items.parquet")
shipments = pd.read_parquet(PROCESSED_DIR / "shipments.parquet")

# Date handling
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['year_month'] = orders['order_date'].dt.to_period('M').astype(str)

print("Data loaded and dates processed.")

Loading datasets...
Data loaded and dates processed.


## 1. Geographic Breakdown
Did all regions collapse at once?

In [15]:
# Merge with geo on ZIP
orders_geo = orders.merge(geo, on='zip')
geo_trend = orders_geo.groupby(['year_month', 'region'])['order_id'].count().reset_index()

fig = px.line(geo_trend, x='year_month', y='order_id', color='region', 
              title='Order Volume Trend by Region (2012-2022)')
fig.write_image(PLOT_DIR / "18_regional_order_trend.png")
fig.show()

## 2. Price Elasticity Analysis
How does average price affect unit sales?

In [16]:
# Merge items with orders to get year_month
it_orders = order_items.merge(orders[['order_id', 'year_month']], on='order_id')

price_trend = it_orders.groupby('year_month').agg(
    avg_price=('unit_price', 'mean'),
    total_units=('quantity', 'sum')
).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=price_trend['year_month'], y=price_trend['avg_price'],
                         mode='lines', name='Avg Unit Price', line=dict(color='purple')))

fig.add_trace(go.Scatter(x=price_trend['year_month'], y=price_trend['total_units'],
                         mode='lines', name='Total Units Sold', yaxis='y2', line=dict(color='orange')))

fig.update_layout(
    title='Price Elasticity: Price Trend vs Units Sold',
    yaxis=dict(title='Avg Price ($)'),
    yaxis2=dict(title='Units Sold', overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99)
)
fig.write_image(PLOT_DIR / "19_price_elasticity.png")
fig.show()

## 3. Shipping Cost Impact
Did expensive shipping kill the conversion?

In [17]:
# Merge shipments with orders to get year_month
ship_orders = shipments.merge(orders[['order_id', 'year_month']], on='order_id')

ship_cost = ship_orders.groupby('year_month')['shipping_fee'].mean().reset_index()
fig = px.line(ship_cost, x='year_month', y='shipping_fee', 
              title='Avg Shipping Fee Trend (2012-2022)')
fig.write_image(PLOT_DIR / "20_shipping_cost_trend.png")
fig.show()